In [2]:
from groq import Groq
from dotenv import load_dotenv
import os
import chromadb
from sentence_transformers import SentenceTransformer

load_dotenv()
client = Groq(api_key = os.getenv("GROQ_API_KEY"))

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("All imports successfull")
print("Embedder loaded:", embedder)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

All imports successfull
Embedder loaded: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [3]:
# Knowledge base about a fictional tech company "NovaTech"
documents = [
    "NovaTech was founded in 2015 and is headquartered in Bangalore, India.",
    "NovaTech's flagship product is an AI-powered customer support platform called HelpDesk Pro.",
    "HelpDesk Pro uses natural language processing to automatically resolve 70 percent of customer queries without human intervention.",
    "NovaTech has a team of 250 employees across engineering, sales, and operations departments.",
    "NovaTech raised 15 million dollars in Series B funding in 2023 led by Sequoia Capital India.",
    "NovaTech's engineering team works with Python, FastAPI, React, PostgreSQL, and AWS.",
    "NovaTech offers three pricing plans: Starter at 29 dollars per month, Growth at 99 dollars per month, and Enterprise with custom pricing.",
    "NovaTech's CEO is Rahul Mehta, who previously worked at Google and Flipkart before founding the company.",
    "NovaTech won the Best AI Startup award at TechSparks 2023 and was featured in Forbes India 30 Under 30.",
    "NovaTech's HelpDesk Pro integrates with Slack, Zendesk, Salesforce, and WhatsApp Business."
]

doc_ids = [f"doc_{i}" for i in range(len(documents))]

print(f"Knowledge base created with {len(documents)} documents")
print(f"Sample document: {documents[0]}")
print(f"Sample ID: {doc_ids[0]}")

Knowledge base created with 10 documents
Sample document: NovaTech was founded in 2015 and is headquartered in Bangalore, India.
Sample ID: doc_0


In [5]:
embeddings = embedder.encode(documents)

print(f"Number of embeddings: {len(embeddings)}")
print(f"Shape of one embedding: {embeddings[0].shape}")
print(f"First 5 numbers of doc_0 embedding: {embeddings[0][:5]}")
print(f"First 5 numbers of doc_1 embedding: {embeddings[1][:5]}")

Number of embeddings: 10
Shape of one embedding: (384,)
First 5 numbers of doc_0 embedding: [-0.00461056 -0.03675355  0.01327296 -0.04586574 -0.06754437]
First 5 numbers of doc_1 embedding: [-0.14127144  0.00913019  0.01219083 -0.06637855  0.02698712]


In [7]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(name="novatech_knowledge")

collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    ids=doc_ids
)

print(f"Collection created: {collection.name}")
print(f"Total documents stored: {collection.count()}")

Collection created: novatech_knowledge
Total documents stored: 10


In [10]:
query = "What programming languages does NoveTech use?"

query_embedding = embedder.encode([query])

results = collection.query(
    query_embeddings = query_embedding.tolist(),
    n_results=2
)

print(f"Query: {query}")
print(f"\nTop 2 most relevant documents found:")
print(f"\n1. {results['documents'][0][0]}")
print(f"   Distance: {results['distances'][0][0]:.4f}")
print(f"\n2. {results['documents'][0][1]}")
print(f"   Distance: {results['distances'][0][1]:.4f}")

Query: What programming languages does NoveTech use?

Top 2 most relevant documents found:

1. NovaTech's engineering team works with Python, FastAPI, React, PostgreSQL, and AWS.
   Distance: 1.1565

2. HelpDesk Pro uses natural language processing to automatically resolve 70 percent of customer queries without human intervention.
   Distance: 1.4433


In [15]:
def rag_answer(question, n_results=2):
    question_embedding = embedder.encode([question])
    results = collection. query(
        query_embeddings=question_embedding.tolist(),
        n_results=n_results
    )

    retrieved_docs=results['documents'][0]
    context="\n".join([f"- {doc}" for doc in retrieved_docs])

    prompt = f"""Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have that information."
Do not use any outside knowledge.

Context:
{context}

Question: {question}

Answer:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return {
        "question": question,
        "retrieved_context": context,
        "answer": response.choices[0].message.content
    }

print("RAG function ready")

RAG function ready


In [16]:
# Test 1 — question that exists in knowledge base
result1 = rag_answer("Who is the CEO of NovaTech?")

print("=" * 50)
print(f"QUESTION: {result1['question']}")
print(f"\nRETRIEVED CONTEXT:\n{result1['retrieved_context']}")
print(f"\nFINAL ANSWER: {result1['answer']}")
print("=" * 50)

QUESTION: Who is the CEO of NovaTech?

RETRIEVED CONTEXT:
- NovaTech's CEO is Rahul Mehta, who previously worked at Google and Flipkart before founding the company.
- NovaTech was founded in 2015 and is headquartered in Bangalore, India.

FINAL ANSWER: Rahul Mehta.


In [17]:
# Test 2 — question NOT in knowledge base
result2 = rag_answer("What is NovaTech's annual revenue?")

print("=" * 50)
print(f"QUESTION: {result2['question']}")
print(f"\nRETRIEVED CONTEXT:\n{result2['retrieved_context']}")
print(f"\nFINAL ANSWER: {result2['answer']}")
print("=" * 50)

print()

# Test 3 — compare RAG vs no RAG on same question
question = "What integrations does HelpDesk Pro support?"

# WITHOUT RAG — LLM answers from training data only
response_no_rag = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": question}],
    temperature=0
)

# WITH RAG — LLM answers from your knowledge base
result3 = rag_answer(question)

print("=" * 50)
print(f"QUESTION: {question}")
print(f"\nWITHOUT RAG:")
print(response_no_rag.choices[0].message.content)
print(f"\nWITH RAG:")
print(f"Context used: {result3['retrieved_context']}")
print(f"Answer: {result3['answer']}")
print("=" * 50)

QUESTION: What is NovaTech's annual revenue?

RETRIEVED CONTEXT:
- NovaTech offers three pricing plans: Starter at 29 dollars per month, Growth at 99 dollars per month, and Enterprise with custom pricing.
- NovaTech raised 15 million dollars in Series B funding in 2023 led by Sequoia Capital India.

FINAL ANSWER: I don't have that information.

QUESTION: What integrations does HelpDesk Pro support?

WITHOUT RAG:
HelpDesk Pro supports various integrations to enhance its functionality and streamline workflows. Some of the integrations it supports include:

1. Email Integration: HelpDesk Pro can be integrated with email services like Gmail, Outlook, and others, allowing users to convert emails into tickets and manage them within the system.
2. CRM Integration: It can be integrated with customer relationship management (CRM) systems like Salesforce, Zoho CRM, and others, enabling users to access customer information and history from within the help desk.
3. E-commerce Integration: HelpDesk

RAG from scratch — key observations:

RETRIEVAL: Query → vector → ChromaDB finds closest documents
AUGMENTATION: Retrieved docs become context in the prompt  
GENERATION: LLM answers from context only, not training data

Why RAG works:
- Without RAG: model hallucinates 10 fake integrations
- With RAG: model gives 4 correct integrations from knowledge base
- "I don't have that information" = hallucination prevention working

Distance threshold matters:
- Low distance = highly relevant document
- High distance = weakly relevant, may need to be discarded
- Production systems filter results above a distance threshold

This is what LangChain's RAG chain does internally —
same pipeline, just with abstractions on top.

In [18]:
# Chunking experiment — why chunk size matters in real RAG systems

long_document = """
NovaTech was founded in 2015 by Rahul Mehta in Bangalore. 
The company started with just 5 employees working on AI solutions.
In 2017, NovaTech launched HelpDesk Pro, their flagship product.
HelpDesk Pro uses NLP to resolve customer queries automatically.
By 2019, NovaTech had grown to 100 employees and expanded to Mumbai.
In 2021, they integrated with Slack, Zendesk and Salesforce.
In 2023, NovaTech raised 15 million dollars in Series B funding.
Today NovaTech has 250 employees and serves 500 plus enterprise clients.
"""

# Small chunks
small_chunks = [s.strip() for s in long_document.strip().split('\n') if s.strip()]

# Large chunk — entire document as one piece
large_chunk = [long_document.strip()]

print(f"Small chunks ({len(small_chunks)} total):")
for i, chunk in enumerate(small_chunks):
    print(f"  Chunk {i}: {chunk[:60]}...")

print(f"\nLarge chunk (1 total):")
print(f"  Chunk 0: {large_chunk[0][:60]}...")

print("\nKey insight:")
print("Small chunks = precise retrieval, less context per chunk")
print("Large chunks = more context, but retrieval less precise")
print("Real RAG systems chunk at 256-512 tokens with 20% overlap")

Small chunks (8 total):
  Chunk 0: NovaTech was founded in 2015 by Rahul Mehta in Bangalore....
  Chunk 1: The company started with just 5 employees working on AI solu...
  Chunk 2: In 2017, NovaTech launched HelpDesk Pro, their flagship prod...
  Chunk 3: HelpDesk Pro uses NLP to resolve customer queries automatica...
  Chunk 4: By 2019, NovaTech had grown to 100 employees and expanded to...
  Chunk 5: In 2021, they integrated with Slack, Zendesk and Salesforce....
  Chunk 6: In 2023, NovaTech raised 15 million dollars in Series B fund...
  Chunk 7: Today NovaTech has 250 employees and serves 500 plus enterpr...

Large chunk (1 total):
  Chunk 0: NovaTech was founded in 2015 by Rahul Mehta in Bangalore. 
T...

Key insight:
Small chunks = precise retrieval, less context per chunk
Large chunks = more context, but retrieval less precise
Real RAG systems chunk at 256-512 tokens with 20% overlap
